# 🧪 Topic 03: Experiment Tracking & Model Registry with MLflow

## 1. Concepts & Architecture
Experiment tracking records all metadata associated with machine learning model training runs:
- **Parameters:** Hyperparameters (`learning_rate`, `n_estimators`, `max_depth`).
- **Metrics:** Performance evaluation scores (`accuracy`, `f1_score`, `rmse`).
- **Artifacts:** Saved model objects (`.pkl`, `.onnx`), plots, confusion matrices.
- **Model Registry:** Central repository to manage model lifecycle states (`Staging`, `Production`, `Archived`).

---

## 2. Hands-on: MLflow Training & Registry Workflow


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import os

mlflow.set_tracking_uri("sqlite:///mlflow_demo.db")
mlflow.set_experiment("MLOps_Crash_Course_Experiment")

X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

experiments = [
    {"n_estimators": 10, "max_depth": 3},
    {"n_estimators": 50, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10}
]

best_acc = 0
best_run_id = None

for params in experiments:
    with mlflow.start_run() as run:
        mlflow.log_params(params)
        
        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        
        mlflow.sklearn.log_model(model, artifact_path="model")
        
        print(f"Run ID: {run.info.run_id[:8]} | Params: {params} | Accuracy: {acc:.4f} | F1: {f1:.4f}")
        
        if acc > best_acc:
            best_acc = acc
            best_run_id = run.info.run_id

print(f"\n🏆 Best Run ID: {best_run_id} with Accuracy: {best_acc:.4f}")
